> 🤖 **AI-Assisted** — Setup and boilerplate imports.

In [1]:
import torch 
import torch.nn as nn 
import torch.nn.functional as F
import math
import time

> ✍️ **My Work** — Figuring out how to use `torch.gather` for embeddings.

In [2]:
class GatherEmbedding(nn.Module): 
    def __init__(self, d_in= 10000, d_out=512):
        super().__init__()
        self.W = nn.Parameter(
                        torch.randn(
                                size = (d_in, d_out),
                                requires_grad=True, 
                                dtype = torch.float
                        ))
        self.d_in = d_in
        self.d_out = d_out 
    
    def forward(self, x): 
        batch_size, seq_len = x.shape

        # get the values for the rows we want and duplicate them d_out times to get all of the values 
        grabber = x.flatten().unsqueeze(1).expand( -1, self.d_out )
        # gather then go back to 3d 
        return self.W.gather(dim = 0, index = grabber).reshape(batch_size, seq_len, self.d_out)
                


> ✍️ **My Work** — Figuring out `torch.sparse_coo` for sparse operations.

In [3]:
class SparseEmbedding(nn.Module): 
    def __init__(self, d_in= 10000, d_out=512):
        super().__init__()
        self.W = nn.Parameter(
                        torch.randn(
                                size = (d_in, d_out),
                                requires_grad=True, 
                                dtype = torch.float
                        ))
        self.d_in = d_in
    
    def forward(self, x): 
        batch_size, seq_len = x.shape

        row_idx = torch.arange(0,batch_size*seq_len)
       
        idx = torch.stack([row_idx.flatten(),x.flatten()])

        vals = torch.ones((batch_size*  seq_len))
         #squash to batch_size*seq_len, self.d_in
        one_hot_sparse = torch.sparse_coo_tensor(idx, vals, size = (batch_size*seq_len,self.d_in ))
        # multiply in 2D
        #(batch* seq_len,v) x (v, d_model) -> (batch * seq_len, d_model)
        O= one_hot_sparse @ self.W
        # reconstruct 
        
        return O.reshape(batch_size, seq_len , -1)


> ✍️ **My Work** — Reimplementing one-hot dense operations for baseline comparison.

In [4]:
class OneHotEmbedding(nn.Module): 
    def __init__(self, d_in= 10000, d_out=512):
        super().__init__()
        self.W = nn.Parameter(
                        torch.randn(
                                size = (d_in, d_out),
                                requires_grad=True, 
                                dtype = torch.float
                        ))
        self.d_in = d_in
    def forward(self, x): 
        #encode in one hot to access embedding vector
        x= F.one_hot(x, num_classes=self.d_in).to(torch.float)
        #(batch, v) x (v, d_model)
        return x.matmul(self.W)

> ✍️ **My Work** — Reimplementing native indexing the right way.

In [5]:
class NativeEmbedding(nn.Module): 
    def __init__(self, d_in=10000, d_out=512):
        super().__init__()
        self.W = nn.Parameter(torch.randn(d_in, d_out, requires_grad=True))
    
    def forward(self, x): 
        return self.W[x]


> 🤖 **AI-Assisted** — Generating the benchmarking loop for performance comparison.

In [7]:


def run_four_way_benchmark():
    # Force to CPU to ensure all 4 backends run reliably without missing GPU kernels
    device = torch.device("cpu")
    print(f"Running 4-Way Forward & Backward Benchmark on: {device.type.upper()}\n")

    # Dimensions (Scaled up to see structural differences clearly)
    vocab_size = 10000
    embed_dim = 256
    batch_size = 64
    seq_len = 256
    iterations = 100  # Reduced slightly because Dense One-Hot is highly demanding

    print(f"Configuration -> Vocab: {vocab_size}, Embed: {embed_dim}, Batch: {batch_size}, Seq: {seq_len}")
    print("-" * 70)

    # 1. Initialize all 4 models (Assumes they exist in your current namespace)
    try:
        model_onehot = OneHotEmbedding(vocab_size, embed_dim).to(device)
        model_sparse = SparseEmbedding(vocab_size, embed_dim).to(device)
        model_gather = GatherEmbedding(vocab_size, embed_dim).to(device)
        model_native = NativeEmbedding(vocab_size, embed_dim).to(device)
    except NameError as e:
        print(f"Initialization Error: Ensure all 4 classes are defined before running. Detail: {e}")
        return

    # 2. Synchronize all weights perfectly to ensure identical math
    shared_weights = model_native.W.data.clone()
    model_onehot.W.data = shared_weights.clone()
    model_sparse.W.data = shared_weights.clone()
    model_gather.W.data = shared_weights.clone()

    # 3. Generate baseline tokens
    x = torch.randint(0, vocab_size, (batch_size, seq_len), device=device)

    # Dictionary to keep track of execution profiles
    profiles = {
        "One-Hot Dense": {"model": model_onehot, "fw": 0.0, "bw": 0.0},
        "Sparse Math":   {"model": model_sparse, "fw": 0.0, "bw": 0.0},
        "Manual Gather": {"model": model_gather, "fw": 0.0, "bw": 0.0},
        "Native Index":  {"model": model_native, "fw": 0.0, "bw": 0.0}
    }

    # 4. Execute Benchmark Loop
    for name, profile in profiles.items():
        model = profile["model"]
        
        # --- FORWARD PASS TIME ---
        start_fw = time.perf_counter()
        for _ in range(iterations):
            # We track gradients here because we want an accurate forward profile 
            # that prepares the autograd graph for the backward pass
            out = model(x)
        profile["fw"] = (time.perf_counter() - start_fw) / iterations

        # --- BACKWARD PASS TIME ---
        start_bw = time.perf_counter()
        for _ in range(iterations):
            model.W.grad = None  # Reset tracking
            out = model(x)
            loss = out.sum()
            loss.backward()
        profile["bw"] = (time.perf_counter() - start_bw) / iterations
        
        print(f"Finished profiling: {name}")

    # 5. Display Performance Leaderboard
    print("\n" + "="*70)
    print(f"{'EMBEDDING PARADIGM':<20} | {'FORWARD AVG (s)':<18} | {'BACKWARD AVG (s)':<18}")
    print("="*70)
    for name, timings in profiles.items():
        print(f"{name:<20} | {timings['fw']:.6f}s            | {timings['bw']:.6f}s")
    print("="*70)

if __name__ == "__main__":
    run_four_way_benchmark()

Running 4-Way Forward & Backward Benchmark on: CPU

Configuration -> Vocab: 10000, Embed: 256, Batch: 64, Seq: 256
----------------------------------------------------------------------
Finished profiling: One-Hot Dense
Finished profiling: Sparse Math
Finished profiling: Manual Gather
Finished profiling: Native Index

EMBEDDING PARADIGM   | FORWARD AVG (s)    | BACKWARD AVG (s)  
One-Hot Dense        | 0.081280s            | 0.126895s
Sparse Math          | 0.001011s            | 0.003414s
Manual Gather        | 0.001068s            | 0.002620s
Native Index         | 0.000420s            | 0.002162s
